# Predicting AQI - v2 (Leaderboard Push)

This notebook is platform-ready:
- Input: `dataset/public/train.csv`, `dataset/public/test.csv`, `dataset/public/sample_submission.csv`
- Output: `working/submission.csv`

Key improvements:
- Strict chronological expanding CV (tail-heavy)
- Leakage-safe fold target statistics
- Multi-seed LightGBM + log-target branch
- ExtraTrees/HistGB support models
- OOF ensemble weight search with tail emphasis
- Station-wise linear calibration (shrunk)
- Conservative high-tail post-processing


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer

from lightgbm import LGBMRegressor, early_stopping, log_evaluation

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def locate_data_dir():
    candidates = [
        'dataset/public',
        './dataset/public',
        '/kaggle/input/predicting-air-quality-index',
        '/kaggle/input/predicting-air-quality-index-dataset',
        '/Users/songling/Desktop/Predicting Air Quality Index',
    ]
    for d in candidates:
        if all(os.path.exists(os.path.join(d, f)) for f in ['train.csv', 'test.csv', 'sample_submission.csv']):
            return d
    raise FileNotFoundError('Cannot locate train/test/sample_submission files.')


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


In [ ]:
DATA_DIR = locate_data_dir()
print('Using DATA_DIR:', DATA_DIR)

train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print('train:', train.shape, 'test:', test.shape)


In [ ]:
def base_features(df):
    out = df.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce')
    out['ts'] = pd.to_datetime(dict(year=out['year'], month=out['month'], day=out['day'])) + pd.to_timedelta(out['hour'], unit='h')

    out['date_ordinal'] = out['date'].map(lambda x: x.toordinal() if pd.notna(x) else np.nan)
    out['dayofyear'] = out['date'].dt.dayofyear.astype(float)
    out['weekofyear'] = out['date'].dt.isocalendar().week.astype(float)
    out['quarter'] = out['date'].dt.quarter.astype(float)

    dow_map = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4,'Saturday':5,'Sunday':6}
    out['dow_num'] = out['day_of_week'].map(dow_map).fillna(0).astype(float)

    out['hour_sin'] = np.sin(2*np.pi*out['hour']/24.0)
    out['hour_cos'] = np.cos(2*np.pi*out['hour']/24.0)
    out['month_sin'] = np.sin(2*np.pi*out['month']/12.0)
    out['month_cos'] = np.cos(2*np.pi*out['month']/12.0)
    out['dow_sin'] = np.sin(2*np.pi*out['dow_num']/7.0)
    out['dow_cos'] = np.cos(2*np.pi*out['dow_num']/7.0)
    out['doy_sin'] = np.sin(2*np.pi*out['dayofyear']/365.25)
    out['doy_cos'] = np.cos(2*np.pi*out['dayofyear']/365.25)

    out['temp_humidity'] = out['temperature'] * out['humidity']
    out['temp_wind'] = out['temperature'] * out['wind_speed']
    out['hum_wind'] = out['humidity'] * out['wind_speed']
    out['wind_vis_ratio'] = out['wind_speed'] / (out['visibility'] + 1e-3)
    out['temp_hum_ratio'] = out['temperature'] / (out['humidity'] + 1e-3)
    out['visibility_inv'] = 1.0 / (out['visibility'] + 1e-3)

    out['is_night'] = out['hour'].isin([0,1,2,3,4,5,22,23]).astype(int)
    out['is_rush_hour'] = out['hour'].isin([7,8,9,17,18,19]).astype(int)

    out['city_station'] = out['city'].astype(str) + '_' + out['station'].astype(str)
    out['station_hour'] = out['station'].astype(str) + '_' + out['hour'].astype(str)
    out['station_month'] = out['station'].astype(str) + '_' + out['month'].astype(str)
    out['city_hour'] = out['city'].astype(str) + '_' + out['hour'].astype(str)
    out['city_season'] = out['city'].astype(str) + '_' + out['season'].astype(str)

    out = out.drop(columns=['date'])
    return out


def smoothed_stat(ref_df, ref_y, apply_df, col, m=120):
    gm = float(ref_y.mean())
    tmp = ref_df[[col]].copy()
    tmp['_y'] = ref_y.values
    agg = tmp.groupby(col)['_y'].agg(['mean', 'count', 'median', 'std'])

    smooth_mean = (agg['mean'] * agg['count'] + gm * m) / (agg['count'] + m)

    mean_map = apply_df[col].map(smooth_mean).fillna(gm)
    cnt_map = apply_df[col].map(agg['count']).fillna(0.0)
    med_map = apply_df[col].map(agg['median']).fillna(gm)
    std_map = apply_df[col].map(agg['std']).fillna(0.0)
    return mean_map, cnt_map, med_map, std_map


def add_fold_target_features(ref_x, ref_y, apply_x):
    ref = ref_x.copy()
    ap = apply_x.copy()
    keys = [
        'station','city','season','day_of_week','hour','month',
        'city_station','station_hour','station_month','city_hour','city_season'
    ]
    for k in keys:
        m, c, med, s = smoothed_stat(ref, ref_y, ap, k, m=120)
        ap[f'te_mean_{k}'] = m
        ap[f'te_cnt_{k}'] = np.log1p(c)
        ap[f'te_med_{k}'] = med
        ap[f'te_std_{k}'] = s

    # One extra trend stat by date_ordinal bucket
    bucket_ref = (ref['date_ordinal'] // 7).astype('int64')
    bucket_ap = (ap['date_ordinal'] // 7).astype('int64')
    tmp = pd.DataFrame({'b': bucket_ref.values, 'y': ref_y.values})
    bmean = tmp.groupby('b')['y'].mean()
    gm = float(ref_y.mean())
    ap['te_week_bucket_mean'] = bucket_ap.map(bmean).fillna(gm)

    return ap


In [ ]:
train_fe = base_features(train)
test_fe = base_features(test)

y = train_fe['aqi'].astype(float)
X_all = train_fe.drop(columns=['aqi']).copy()
X_test_all = test_fe.copy()

# Keep ids for final output only
train_ids = X_all['id'].astype(int).values
test_ids = X_test_all['id'].astype(int).values
X_all = X_all.drop(columns=['id'])
X_test_all = X_test_all.drop(columns=['id'])

# Chronological expanding splits; tail-heavy (last folds represent future years)
order = np.argsort(train_fe['ts'].values)
parts = np.array_split(order, 7)  # 6 folds
splits = []
for i in range(1, len(parts)):
    tr_idx = np.concatenate(parts[:i])
    va_idx = parts[i]
    if len(tr_idx) > 0 and len(va_idx) > 0:
        splits.append((tr_idx, va_idx))

print('CV folds:', len(splits))
for i, (tr_idx, va_idx) in enumerate(splits, 1):
    print(f'Fold {i}: train={len(tr_idx)}, valid={len(va_idx)}')


In [ ]:
# Model set
lgb_base_params = dict(
    objective='regression',
    metric='l2',
    n_estimators=4000,
    learning_rate=0.012,
    num_leaves=256,
    min_child_samples=15,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.08,
    reg_lambda=1.35,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

lgb_seed_list = [42, 2024, 3407]

cat_cols = ['day_of_week','season','city','station','city_station','station_hour','station_month','city_hour','city_season']
num_cols = [c for c in X_all.columns if c not in cat_cols + ['ts']]

# sklearn branch
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', make_ohe())]), cat_cols)
], remainder='drop')

etr_template = TransformedTargetRegressor(
    regressor=Pipeline([
        ('pre', pre),
        ('reg', ExtraTreesRegressor(
            n_estimators=800,
            max_features=0.8,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),
    transformer=PowerTransformer(method='yeo-johnson', standardize=True)
)

hgb_template = TransformedTargetRegressor(
    regressor=Pipeline([
        ('pre', pre),
        ('reg', HistGradientBoostingRegressor(
            learning_rate=0.025,
            max_depth=12,
            max_iter=1200,
            min_samples_leaf=18,
            l2_regularization=0.2,
            random_state=RANDOM_STATE
        ))
    ]),
    transformer=PowerTransformer(method='yeo-johnson', standardize=True)
)


In [ ]:
# OOF and test predictions
model_oof = {}
model_test = {}

# Prior baseline model (hierarchical mean)
def prior_predict(ref_x, ref_y, query_x):
    gm = float(ref_y.mean())
    ref = ref_x.copy()
    ref['y'] = ref_y.values

    def map_mean_cnt(key):
        g = ref.groupby(key)['y'].agg(['mean', 'count'])
        p = query_x[key].map(g['mean']).fillna(gm)
        c = query_x[key].map(g['count']).fillna(0)
        return p, c

    p1, c1 = map_mean_cnt('station_hour')
    p2, c2 = map_mean_cnt('station')
    p3, c3 = map_mean_cnt('city_hour')
    p4, c4 = map_mean_cnt('city')

    w1 = np.log1p(c1)
    w2 = np.log1p(c2)
    w3 = np.log1p(c3)
    w4 = np.log1p(c4)
    ws = w1 + w2 + w3 + w4 + 1e-9
    pred = (w1*p1 + w2*p2 + w3*p3 + w4*p4) / ws
    return pred.values

# LightGBM raw-target (multi-seed)
oof_lgb_raw = np.zeros(len(X_all), dtype=float)
test_lgb_raw_folds = []
for tr_idx, va_idx in splits:
    Xtr0 = X_all.iloc[tr_idx].drop(columns=['ts']).copy()
    ytr = y.iloc[tr_idx].copy()
    Xva0 = X_all.iloc[va_idx].drop(columns=['ts']).copy()

    Xtr = add_fold_target_features(Xtr0, ytr, Xtr0)
    Xva = add_fold_target_features(Xtr0, ytr, Xva0)
    Xte = add_fold_target_features(Xtr0, ytr, X_test_all.drop(columns=['ts']).copy())

    for c in cat_cols:
        Xtr[c] = Xtr[c].astype('category')
        Xva[c] = Xva[c].astype('category')
        Xte[c] = Xte[c].astype('category')

    fold_va_preds = []
    fold_te_preds = []
    for sd in lgb_seed_list:
        params = dict(lgb_base_params)
        params['random_state'] = sd
        mdl = LGBMRegressor(**params)
        mdl.fit(
            Xtr, ytr,
            eval_set=[(Xva, y.iloc[va_idx])],
            eval_metric='l2',
            callbacks=[early_stopping(200, verbose=False), log_evaluation(0)]
        )
        fold_va_preds.append(mdl.predict(Xva, num_iteration=mdl.best_iteration_))
        fold_te_preds.append(mdl.predict(Xte, num_iteration=mdl.best_iteration_))

    oof_lgb_raw[va_idx] = np.mean(np.vstack(fold_va_preds), axis=0)
    test_lgb_raw_folds.append(np.mean(np.vstack(fold_te_preds), axis=0))

model_oof['lgb_raw'] = oof_lgb_raw
model_test['lgb_raw'] = np.mean(np.vstack(test_lgb_raw_folds), axis=0)

# LightGBM log-target branch
oof_lgb_log = np.zeros(len(X_all), dtype=float)
test_lgb_log_folds = []
for tr_idx, va_idx in splits:
    Xtr0 = X_all.iloc[tr_idx].drop(columns=['ts']).copy()
    ytr = y.iloc[tr_idx].copy()
    Xva0 = X_all.iloc[va_idx].drop(columns=['ts']).copy()

    Xtr = add_fold_target_features(Xtr0, ytr, Xtr0)
    Xva = add_fold_target_features(Xtr0, ytr, Xva0)
    Xte = add_fold_target_features(Xtr0, ytr, X_test_all.drop(columns=['ts']).copy())

    for c in cat_cols:
        Xtr[c] = Xtr[c].astype('category')
        Xva[c] = Xva[c].astype('category')
        Xte[c] = Xte[c].astype('category')

    ytr_log = np.log1p(ytr)
    fold_va_preds = []
    fold_te_preds = []
    for sd in lgb_seed_list:
        params = dict(lgb_base_params)
        params['random_state'] = sd + 100
        mdl = LGBMRegressor(**params)
        mdl.fit(
            Xtr, ytr_log,
            eval_set=[(Xva, np.log1p(y.iloc[va_idx]))],
            eval_metric='l2',
            callbacks=[early_stopping(200, verbose=False), log_evaluation(0)]
        )
        fold_va_preds.append(np.expm1(mdl.predict(Xva, num_iteration=mdl.best_iteration_)))
        fold_te_preds.append(np.expm1(mdl.predict(Xte, num_iteration=mdl.best_iteration_)))

    oof_lgb_log[va_idx] = np.mean(np.vstack(fold_va_preds), axis=0)
    test_lgb_log_folds.append(np.mean(np.vstack(fold_te_preds), axis=0))

model_oof['lgb_log'] = oof_lgb_log
model_test['lgb_log'] = np.mean(np.vstack(test_lgb_log_folds), axis=0)

# ExtraTrees + HistGB
for name, template in [('etr', etr_template), ('hgb', hgb_template)]:
    oof = np.zeros(len(X_all), dtype=float)
    test_fold_preds = []
    for tr_idx, va_idx in splits:
        Xtr = X_all.iloc[tr_idx].drop(columns=['ts']).copy()
        ytr = y.iloc[tr_idx].copy()
        Xva = X_all.iloc[va_idx].drop(columns=['ts']).copy()
        Xte = X_test_all.drop(columns=['ts']).copy()

        mdl = template
        mdl.fit(Xtr, ytr)
        oof[va_idx] = mdl.predict(Xva)
        test_fold_preds.append(mdl.predict(Xte))

    model_oof[name] = oof
    model_test[name] = np.mean(np.vstack(test_fold_preds), axis=0)

# Prior model
oof_prior = np.zeros(len(X_all), dtype=float)
prior_test_folds = []
for tr_idx, va_idx in splits:
    Xtr = X_all.iloc[tr_idx].drop(columns=['ts']).copy()
    ytr = y.iloc[tr_idx].copy()
    Xva = X_all.iloc[va_idx].drop(columns=['ts']).copy()
    Xte = X_test_all.drop(columns=['ts']).copy()

    oof_prior[va_idx] = prior_predict(Xtr, ytr, Xva)
    prior_test_folds.append(prior_predict(Xtr, ytr, Xte))

model_oof['prior'] = oof_prior
model_test['prior'] = np.mean(np.vstack(prior_test_folds), axis=0)

for k, p in model_oof.items():
    mask = p != 0
    print(k, 'OOF MSE =', round(mean_squared_error(y[mask], p[mask]), 6))


In [ ]:
# Tail-emphasized blend optimization
names = list(model_oof.keys())
P = np.column_stack([model_oof[n] for n in names])
Pt = np.column_stack([model_test[n] for n in names])

valid_mask = np.any(P != 0, axis=1)
Pv = P[valid_mask]
yv = y.values[valid_mask]
station_v = train['station'].iloc[np.where(valid_mask)[0]].astype(str).values

# Tail-weighted objective to better handle large AQI errors
q75 = np.quantile(yv, 0.75)
w_tail = np.where(yv >= q75, 2.0, 1.0)


def weighted_mse(y_true, y_pred, w):
    e2 = (y_true - y_pred) ** 2
    return float(np.sum(w * e2) / np.sum(w))

best = (1e18, None)
rng = np.random.default_rng(RANDOM_STATE)
for _ in range(160000):
    w = rng.dirichlet(np.ones(len(names)) * 1.15)
    pred = Pv @ w
    score = weighted_mse(yv, pred, w_tail)
    if score < best[0]:
        best = (score, w)

best_w = best[1]
print('Best weighted OOF score:', best[0])
print('Weights:', {n: float(v) for n, v in zip(names, best_w)})

# Base blend
oof_blend = Pv @ best_w
test_blend = Pt @ best_w

# Station-wise linear calibration on OOF (shrunk)
def fit_station_calibrator(stations, pred, y_true):
    df = pd.DataFrame({'station': stations, 'pred': pred, 'y': y_true})
    global_a = 1.0
    global_b = float(np.mean(y_true - pred))

    cal = {}
    for st, g in df.groupby('station'):
        x = g['pred'].values
        t = g['y'].values
        n = len(g)
        if n < 20:
            cal[st] = (global_a, global_b, n)
            continue
        x_mean = np.mean(x)
        t_mean = np.mean(t)
        var_x = np.mean((x - x_mean) ** 2)
        if var_x < 1e-8:
            a = global_a
        else:
            cov = np.mean((x - x_mean) * (t - t_mean))
            a = cov / (var_x + 1e-8)
        b = t_mean - a * x_mean

        # shrink toward global calibration by sample size
        lam = n / (n + 60.0)
        a = lam * a + (1 - lam) * global_a
        b = lam * b + (1 - lam) * global_b
        cal[st] = (a, b, n)
    return cal, global_a, global_b


cal_map, g_a, g_b = fit_station_calibrator(station_v, oof_blend, yv)

# apply calibrator on test stations
test_stations = test['station'].astype(str).values
pred_cal = np.empty_like(test_blend)
for i, st in enumerate(test_stations):
    a, b, _ = cal_map.get(st, (g_a, g_b, 0))
    pred_cal[i] = a * test_blend[i] + b

# conservative tail shrink to control very large errors
hi = np.quantile(y.values, 0.97)
pred_cal = np.where(pred_cal > hi, hi + 0.985 * (pred_cal - hi), pred_cal)

# final bounds
pred_cal = np.clip(pred_cal, 25.0, 500.0)

submission = pd.DataFrame({'id': test_ids, 'aqi': pred_cal}).sort_values('id').reset_index(drop=True)
os.makedirs('working', exist_ok=True)
submission.to_csv('working/submission.csv', index=False)

print('Saved: working/submission.csv')
print('Submission shape:', submission.shape)
print(submission.head())
